# Dark Matters ABM
## Structured Analysis Workbook

This notebook turns a single exported simulation CSV into a readable analysis report.
It is designed to feel like a guided workbook rather than a raw script, with narrative framing, tables, and one figure per code cell.

### What this workbook helps you do
- inspect the scenario configuration
- summarize the trajectory of the run
- identify formal tipping points
- compare short-term extraction to long-term sustainability
- generate publication-friendly figures with clear interpretation prompts

## Contents

1. Setup and data loading
2. Theory and metric guide
3. Scenario and descriptive summaries
4. Formal tipping-point review
5. Narrative interpretation
6. One-figure-at-a-time diagnostic plots
7. Discussion prompts for writing and presentation

## Suggested workflow

Run the notebook from top to bottom. If you are using Google Colab, leave `CSV_PATH = None` and upload the exported file when prompted. If you are running locally, set `CSV_PATH` to the file you want to analyze.

In [ ]:
from __future__ import annotations

from pathlib import Path
from textwrap import dedent
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.titlesize": 13,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "axes.facecolor": "#fcfcfd",
        "figure.facecolor": "white",
        "axes.edgecolor": "#cbd5e1",
        "grid.color": "#cbd5e1",
        "grid.alpha": 0.35,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


## Load A CSV Export

The notebook supports two modes:

- local/Jupyter: set `CSV_PATH` to a file path
- Google Colab: leave `CSV_PATH = None` and run the cell to upload a file


In [ ]:
CSV_PATH = None


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def load_csv_from_colab_upload() -> pd.DataFrame:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")
    first_name = next(iter(uploaded))
    return pd.read_csv(first_name)


def load_dataframe(csv_path: str | Path | None) -> pd.DataFrame:
    if csv_path is None:
        if in_colab():
            return load_csv_from_colab_upload()
        raise ValueError("Set CSV_PATH when running outside Google Colab.")
    return pd.read_csv(csv_path)


## Theory And Metric Guide

### What the exported CSV represents
Each row corresponds to one simulation step. The exported file repeats the simulation parameters so the run can always be interpreted together with the scenario configuration.

### Formal tipping-point detection
A tipping point is only recorded when a threshold is sustained for three consecutive steps.

The current rules are:

- Trust Collapse: mean trust is at or below `0.50`
- Social Contagion: negative WOM is at or above `0.22`
- Churn Cascade: cumulative churn is at or above `0.35`
- Extractive Divergence: revenue gap exceeds `20%` of short-term revenue while cumulative churn is at or above `0.15`

### Core metrics

- `active_users`: users still active on the platform
- `mean_trust`: average trust among active users
- `mean_harm`: average accumulated harm among active users
- `churn_rate`: step-level increase in cumulative churn
- `cumulative_churn`: share of users who have churned
- `negative_wom_rate`: average negative word-of-mouth among active users
- `reputation`: platform reputation proxy
- `short_term_revenue`: accumulated short-run revenue proxy
- `long_term_revenue`: revenue proxy discounted by churn


In [ ]:
PARAMETER_COLUMNS = [
    "adaptive_platform",
    "alpha_exposure_to_trust",
    "avg_degree",
    "beta_support_recovery",
    "complaint_propensity_mean",
    "complaint_propensity_sd",
    "customer_support_quality",
    "dark_pattern_intensity",
    "delta_exposure_to_harm",
    "digital_literacy_mean",
    "digital_literacy_sd",
    "drip_pricing_exposure_prob",
    "drip_pricing_severity",
    "forced_trial_exposure_prob",
    "forced_trial_severity",
    "gamma_social_trust_loss",
    "hard_cancel_exposure_prob",
    "hard_cancel_severity",
    "manipulation_sensitivity_mean",
    "manipulation_sensitivity_sd",
    "max_steps",
    "network_type",
    "num_users",
    "pattern_drip_pricing",
    "pattern_forced_trial",
    "pattern_hard_cancel",
    "review_visibility",
    "rewire_prob",
    "seed",
    "social_activity_mean",
    "social_activity_sd",
    "social_influence_strength",
    "switching_cost_mean",
    "switching_cost_sd",
    "theta0",
    "theta_harm",
    "theta_social",
    "theta_switching_cost",
    "theta_trust",
    "trust_baseline_mean",
    "trust_baseline_sd",
]

METRIC_COLUMNS = [
    "active_users",
    "any_tipping_point_triggered",
    "churn_rate",
    "cumulative_churn",
    "extractive_divergence_step",
    "extractive_divergence_triggered",
    "first_tipping_point_step",
    "long_term_revenue",
    "mean_harm",
    "mean_trust",
    "negative_wom_rate",
    "reputation",
    "short_term_revenue",
    "social_contagion_step",
    "social_contagion_triggered",
    "step",
    "tipping_points_triggered_count",
    "trust_collapse_step",
    "trust_collapse_triggered",
    "churn_cascade_step",
    "churn_cascade_triggered",
]

TIPPING_POINT_SPECS = {
    "trust_collapse": {
        "trigger_column": "trust_collapse_triggered",
        "step_column": "trust_collapse_step",
        "label": "Trust Collapse",
        "color": "#2563eb",
    },
    "social_contagion": {
        "trigger_column": "social_contagion_triggered",
        "step_column": "social_contagion_step",
        "label": "Social Contagion",
        "color": "#059669",
    },
    "churn_cascade": {
        "trigger_column": "churn_cascade_triggered",
        "step_column": "churn_cascade_step",
        "label": "Churn Cascade",
        "color": "#dc2626",
    },
    "extractive_divergence": {
        "trigger_column": "extractive_divergence_triggered",
        "step_column": "extractive_divergence_step",
        "label": "Extractive Divergence",
        "color": "#d97706",
    },
}

CORRELATION_COLUMNS = [
    "active_users",
    "active_share",
    "churn_rate",
    "cumulative_churn",
    "long_term_revenue",
    "mean_harm",
    "mean_trust",
    "negative_wom_rate",
    "reputation",
    "revenue_gap",
    "short_term_revenue",
    "trust_loss_from_baseline",
]


def show_md(text: str) -> None:
    display(Markdown(text))


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for column in cleaned.columns:
        if cleaned[column].dtype == object:
            lowered = cleaned[column].astype(str).str.lower()
            if lowered.isin({"true", "false"}).all():
                cleaned[column] = lowered.map({"true": True, "false": False})

    for column in METRIC_COLUMNS:
        if column in cleaned.columns:
            cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

    if "step" in cleaned.columns:
        cleaned = cleaned.sort_values("step").reset_index(drop=True)

    if "num_users" in cleaned.columns and "active_users" in cleaned.columns:
        cleaned["active_share"] = cleaned["active_users"] / cleaned["num_users"]

    if "mean_trust" in cleaned.columns:
        baseline = float(cleaned["mean_trust"].iloc[0])
        cleaned["trust_loss_from_baseline"] = baseline - cleaned["mean_trust"]

    if "short_term_revenue" in cleaned.columns and "long_term_revenue" in cleaned.columns:
        cleaned["revenue_gap"] = cleaned["short_term_revenue"] - cleaned["long_term_revenue"]

    return cleaned


def tipping_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for spec in TIPPING_POINT_SPECS.values():
        trigger_column = spec["trigger_column"]
        step_column = spec["step_column"]
        if trigger_column not in df.columns or step_column not in df.columns:
            continue

        triggered = bool(df[trigger_column].fillna(0).max() >= 1)
        raw_step = df.loc[df[step_column] >= 0, step_column]
        trigger_step = None if raw_step.empty else int(raw_step.iloc[0])
        rows.append(
            {
                "tipping_point": spec["label"],
                "triggered": triggered,
                "trigger_step": trigger_step,
            }
        )
    return pd.DataFrame(rows)


def parameter_summary(df: pd.DataFrame) -> pd.DataFrame:
    available = [column for column in PARAMETER_COLUMNS if column in df.columns]
    if not available:
        return pd.DataFrame(columns=["parameter", "value"])
    row = df.iloc[0][available]
    return pd.DataFrame({"parameter": available, "value": row.values})


def metric_summary(df: pd.DataFrame) -> pd.DataFrame:
    available = [column for column in METRIC_COLUMNS if column in df.columns]
    if not available:
        return pd.DataFrame()
    summary = pd.DataFrame(
        {
            "start": df[available].iloc[0],
            "end": df[available].iloc[-1],
            "min": df[available].min(),
            "max": df[available].max(),
            "mean": df[available].mean(),
        }
    )
    return summary.round(4)


def triggered_tipping_labels(df: pd.DataFrame) -> list[str]:
    summary = tipping_summary(df)
    if summary.empty:
        return []
    return summary.loc[summary["triggered"], "tipping_point"].astype(str).tolist()


def narrative_summary(df: pd.DataFrame) -> str:
    final = df.iloc[-1]
    start = df.iloc[0]

    trust_change = float(final["mean_trust"] - start["mean_trust"]) if "mean_trust" in df.columns else 0.0
    churn_end = float(final["cumulative_churn"]) if "cumulative_churn" in df.columns else 0.0
    wom_peak = float(df["negative_wom_rate"].max()) if "negative_wom_rate" in df.columns else 0.0
    harm_end = float(final["mean_harm"]) if "mean_harm" in df.columns else 0.0
    revenue_gap_end = float(final["revenue_gap"]) if "revenue_gap" in df.columns else 0.0
    triggered = triggered_tipping_labels(df)

    if trust_change <= -0.15:
        trust_read = "The run shows a substantial erosion of trust."
    elif trust_change <= -0.05:
        trust_read = "The run shows a moderate decline in trust."
    else:
        trust_read = "Trust remains comparatively stable over the run."

    if churn_end >= 0.5:
        churn_read = "Cumulative churn ends at a high level, suggesting deep long-run user loss."
    elif churn_end >= 0.2:
        churn_read = "Cumulative churn is meaningful and should be treated as a warning signal."
    else:
        churn_read = "Cumulative churn remains comparatively contained."

    if revenue_gap_end > 0:
        revenue_read = (
            "Short-term revenue exceeds long-term revenue at the end of the run, "
            "which is consistent with extraction outpacing durable value."
        )
    else:
        revenue_read = (
            "Short-term and long-term revenue remain relatively aligned, "
            "which suggests less divergence between extraction and sustainability."
        )

    if triggered:
        tipping_read = "The formal tipping detector was activated for: " + ", ".join(f"**{name}**" for name in triggered) + "."
    else:
        tipping_read = "No formal tipping point is triggered under the current detection rules."

    return dedent(
        f"""
        ## Automated Interpretation

        {trust_read}

        {churn_read}

        End-of-run mean harm is **{harm_end:.3f}** and the peak negative WOM rate is **{wom_peak:.3f}**.

        {revenue_read}

        {tipping_read}
        """
    ).strip()


def add_tipping_markers(df: pd.DataFrame, ax) -> None:
    used_labels = set()
    for spec in TIPPING_POINT_SPECS.values():
        step_column = spec["step_column"]
        if step_column not in df.columns:
            continue
        valid_steps = df.loc[df[step_column] >= 0, step_column]
        if valid_steps.empty:
            continue
        trigger_step = int(valid_steps.iloc[0])
        label = spec["label"]
        ax.axvline(
            trigger_step,
            color=spec["color"],
            linestyle="--",
            linewidth=1.6,
            alpha=0.75,
            label=label if label not in used_labels else None,
        )
        used_labels.add(label)


def finish_axis(ax, *, xlabel: str, ylabel: str, title: str) -> None:
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(alpha=0.25)
    sns.despine(ax=ax)


def line_chart(
    df: pd.DataFrame,
    columns: Iterable[str],
    title: str,
    ylabel: str,
    colors: list[str] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[int, int] = (12, 5),
) -> None:
    fig, ax = plt.subplots(figsize=figsize)
    for idx, column in enumerate(columns):
        if column not in df.columns:
            continue
        color = None if colors is None else colors[idx]
        ax.plot(
            df["step"],
            df[column],
            linewidth=2.4,
            label=column,
            color=color,
            marker="o",
            markersize=3,
            alpha=0.95,
        )
    if ylim is not None:
        ax.set_ylim(*ylim)
    add_tipping_markers(df, ax)
    finish_axis(ax, xlabel="Step", ylabel=ylabel, title=title)
    ax.legend(fontsize=9, frameon=True)
    plt.tight_layout()
    plt.show()


## Load, Clean, And Inspect The Run

This cell loads the CSV, cleans booleans and numeric columns, derives helper variables such as `active_share`, `trust_loss_from_baseline`, and `revenue_gap`, and shows the run metadata.


In [ ]:
df = clean_dataframe(load_dataframe(CSV_PATH))

if "simulation_id" in df.columns:
    simulation_ids = df["simulation_id"].dropna().astype(str).unique().tolist()
    show_md(
        f"## Loaded simulation data\n\n"
        f"- Rows: **{len(df)}**\n"
        f"- Simulation IDs found: **{len(simulation_ids)}**\n"
        f"- Selected run for analysis: **{simulation_ids[0]}**"
    )
else:
    show_md(f"## Loaded simulation data\n\n- Rows: **{len(df)}**")

df.head()


## Scenario Parameters

This table captures the configuration of the run. Because the export repeats these fields in every row, we only need the first row to summarize the scenario.


In [ ]:
parameter_summary(df)


## Descriptive Summary Of The Run

The table below compares the start, end, minimum, maximum, and mean of the main simulation metrics across the full run.


In [ ]:
metric_summary(df)


## Formal Tipping-Point Summary

This table shows whether each tipping point was triggered and, if so, the first step at which it was confirmed.


In [ ]:
tipping_summary(df)


## Automated Interpretation

This text block converts the end-state of the run into a short narrative summary. It is not a replacement for interpretation, but it gives a fast read on whether the scenario looks comparatively stable, warning-heavy, or clearly extractive.


In [ ]:
show_md(narrative_summary(df))


## Plot 1: Trust And Reputation

This figure shows whether users continue to trust the platform and whether the platform's aggregate standing remains stable over time. A strong joint decline suggests that manipulation is undermining both user sentiment and system legitimacy.


In [ ]:
line_chart(
    df,
    ["mean_trust", "reputation"],
    "Trust and Reputation Over Time",
    "Value",
    colors=["#2563eb", "#9333ea"],
    ylim=(0, 1),
)


## Plot 2: Churn And Negative WOM

This figure helps identify diffusion and exit dynamics. If churn and negative word-of-mouth rise together, the model is showing a social pathway through which bad experience spreads and then converts into user loss.


In [ ]:
line_chart(
    df,
    ["churn_rate", "cumulative_churn", "negative_wom_rate"],
    "Churn and Negative WOM Over Time",
    "Value",
    colors=["#ef4444", "#f97316", "#059669"],
    ylim=(0, 1),
)


## Plot 3: Revenue Dynamics

This is the key extraction-versus-sustainability plot. The main question is whether short-term revenue keeps rising while long-term revenue falls behind, indicating divergence between immediate gains and durable platform value.


In [ ]:
line_chart(
    df,
    ["short_term_revenue", "long_term_revenue", "revenue_gap"],
    "Revenue Dynamics",
    "Revenue proxy",
    colors=["#0f172a", "#0891b2", "#dc2626"],
)


## Plot 4: Active Users And Mean Harm

This figure compares engagement and accumulated harm. It helps answer whether the platform is maintaining participation while silently accumulating user damage, or whether harm is already converting into visible exit.


In [ ]:
line_chart(
    df,
    ["active_users", "mean_harm"],
    "Active Users and Mean Harm",
    "Value",
    colors=["#16a34a", "#f59e0b"],
)


## Plot 5: Normalized System Trajectories

This plot rescales several core metrics to a common 0-1 range. It is useful when the raw metrics have different units and you want to compare shape, timing, and turning points rather than absolute magnitude.


In [ ]:
columns = [
    "mean_trust",
    "reputation",
    "cumulative_churn",
    "negative_wom_rate",
    "mean_harm",
    "active_share",
]
available = [column for column in columns if column in df.columns]

normalized = df[available].copy()
for column in available:
    col_min = normalized[column].min()
    col_max = normalized[column].max()
    if col_max == col_min:
        normalized[column] = 0.0
    else:
        normalized[column] = (normalized[column] - col_min) / (col_max - col_min)
normalized["step"] = df["step"]

fig, ax = plt.subplots(figsize=(12, 6))
for column in available:
    ax.plot(normalized["step"], normalized[column], linewidth=2.1, label=column, marker="o", markersize=2.8)
add_tipping_markers(df, ax)
finish_axis(ax, xlabel="Step", ylabel="Normalized value", title="Normalized System Trajectories")
ax.legend(ncol=2, fontsize=9, frameon=True)
plt.tight_layout()
plt.show()


## Plot 6: Correlation Heatmap

This is an exploratory diagnostic. It does not prove causation, but it helps show which metrics move together inside the run and whether trust loss, churn, harm, reputation, and revenue separation appear tightly linked.


In [ ]:
columns = [column for column in CORRELATION_COLUMNS if column in df.columns]
columns = [column for column in columns if df[column].nunique(dropna=True) > 1]

corr = df[columns].corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    annot=True,
    annot_kws={"size": 8},
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation Heatmap for Time-Series Metrics", fontsize=13, pad=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()


## Plot 7: Step-To-Step Change Diagnostics

This plot focuses on deltas rather than levels. It is useful for identifying moments of acceleration or reversal, such as sudden increases in churn or abrupt drops in trust.


In [ ]:
delta_columns = [
    column
    for column in [
        "mean_trust",
        "reputation",
        "cumulative_churn",
        "short_term_revenue",
        "long_term_revenue",
    ]
    if column in df.columns
]

delta_df = df[["step", *delta_columns]].copy()
for column in delta_columns:
    delta_df[column] = delta_df[column].diff().fillna(0.0)

fig, ax = plt.subplots(figsize=(12, 6))
for column in delta_columns:
    ax.plot(delta_df["step"], delta_df[column], linewidth=2.0, label=f"d({column})", marker="o", markersize=2.8)
ax.axhline(0, color="#64748b", linewidth=1, alpha=0.7)
add_tipping_markers(df, ax)
finish_axis(ax, xlabel="Step", ylabel="Delta", title="Step-to-Step Change in Key Metrics")
ax.legend(ncol=2, fontsize=9, frameon=True)
plt.tight_layout()
plt.show()


## Suggested Discussion Prompts

- At what point does trust begin to break down more quickly?
- Does negative WOM appear before churn accelerates, or at the same time?
- How large is the final gap between short-term and long-term revenue?
- Does the parameter combination look extractive, sustainable, or mixed?
- If adaptive behavior was enabled, did the platform stabilize outcomes?
